# 01 — Data Exploration

Goals:
1. Load and inspect the MS MARCO dataset
2. Generate float32 embeddings
3. Understand the embedding distribution (mean, std, percentiles)
4. Understand what the baseline FAISS search returns

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Load MS MARCO dataset

In [ ]:
from datasets import load_dataset

LIMIT = 50_000
dataset = load_dataset("ms_marco", "v1.1", split=f"train[:{LIMIT}]")
print(f"Dataset size: {len(dataset)}")
print("\nFirst example:")
print(dataset[0])

In [ ]:
# Extract passages
passages = []
for item in dataset:
    for text in item["passages"]["passage_text"]:
        passages.append(text)
        if len(passages) >= LIMIT:
            break
    if len(passages) >= LIMIT:
        break

print(f"Total passages: {len(passages)}")
print("\nSample passage:")
print(passages[0][:300])

In [ ]:
# Passage length distribution
lengths = [len(p.split()) for p in passages]
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, edgecolor='white', color='steelblue')
plt.xlabel('Passage length (words)')
plt.ylabel('Count')
plt.title('MS MARCO passage length distribution')
plt.axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean: {np.mean(lengths):.0f}')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Mean: {np.mean(lengths):.0f} | Median: {np.median(lengths):.0f} | Max: {max(lengths)}")

## 2. Generate and inspect float32 embeddings

In [ ]:
# Check if embeddings already exist; if not, generate them
emb_path = Path('../embeddings/float32/embeddings.npy')

if emb_path.exists():
    print("Loading pre-computed embeddings...")
    embeddings = np.load(emb_path)
else:
    print("Generating embeddings (this takes a few minutes)...")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embeddings = model.encode(passages, batch_size=64, show_progress_bar=True)
    embeddings = embeddings.astype(np.float32)
    emb_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(emb_path, embeddings)

print(f"Shape: {embeddings.shape}")
print(f"dtype: {embeddings.dtype}")
print(f"Memory: {embeddings.nbytes / 1e6:.1f} MB")

In [ ]:
# Embedding value distribution
flat = embeddings.flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(flat, bins=100, color='steelblue', edgecolor='none', density=True)
axes[0].set_title('Distribution of all embedding values')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')
for d in [0.05, 0.1, 0.2]:
    axes[0].axvline(d, color='red', linestyle=':', alpha=0.7, label=f'δ={d}')
    axes[0].axvline(-d, color='red', linestyle=':', alpha=0.7)
axes[0].legend()

# Per-dimension std
dim_std = embeddings.std(axis=0)
axes[1].hist(dim_std, bins=50, color='coral', edgecolor='none')
axes[1].set_title('Per-dimension std across corpus')
axes[1].set_xlabel('Std')
axes[1].set_ylabel('Count (dimensions)')

plt.tight_layout()
plt.show()

print(f"Global stats — mean: {flat.mean():.4f}, std: {flat.std():.4f}")
print(f"Values in (-0.1, +0.1): {((flat > -0.1) & (flat < 0.1)).mean():.1%}")

## 3. Validate the FAISS baseline

In [ ]:
from src.baseline import FAISSBaseline

baseline = FAISSBaseline()
baseline.add(embeddings)
print(baseline)

In [ ]:
# Sample a query and inspect results
query_idx = 42
query_emb = embeddings[query_idx]

indices, scores = baseline.search(query_emb, k=10)

print(f"Query ({query_idx}): {passages[query_idx][:100]}...")
print("\nTop-10 results:")
for rank, (idx, score) in enumerate(zip(indices, scores), 1):
    print(f"  {rank}. [{idx}] score={score:.3f}  {passages[idx][:80]}...")

In [ ]:
# Cosine score distribution for top results vs. random
top_scores = scores
rand_idx = np.random.choice(len(embeddings), 10)
rand_q = embeddings[rand_idx[0]]
rand_indices, rand_scores = baseline.search(rand_q, k=1000)

plt.figure(figsize=(10, 4))
plt.hist(rand_scores, bins=50, alpha=0.5, label='All scores (top 1000)', color='gray')
plt.axvline(top_scores[0], color='red', linestyle='--', label=f'Top-1 score: {top_scores[0]:.3f}')
plt.axvline(top_scores[-1], color='orange', linestyle='--', label=f'Top-10 score: {top_scores[-1]:.3f}')
plt.xlabel('Cosine similarity')
plt.ylabel('Count')
plt.title('Score distribution: top-1000 results')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Summary

Key observations:
- The corpus is **50k passages** from MS MARCO.
- Embeddings are **384-dimensional float32**, using `all-MiniLM-L6-v2`.
- Float32 index memory: ~**73 MB**.
- Most embedding values fall close to zero — suggesting good ternary compression potential.
- FAISS baseline returns semantically coherent top-10 results.